# CodeBuddy 风格 Agent Loop Demo（deepagents）

把 Coding Agent 的内部循环设计成一份「**有边界的局部控制合同**」（Loop Engineering）：

| 构件 | 作用 |
| --- | --- |
| `local_aim` 局部目标 | 循环收敛的范围，生命周期内不变 |
| `action` 行动策略 | 每次迭代派 agent 行动一步（手段可变，目标不可变）|
| `evaluator` 评测器 | 唯一能宣布「成功」的角色，也可触发升级 |
| `budget` 预算 | 最大迭代次数，耗尽即 `budget_exhausted` |
| `escalation` 升级 | 越权 / 目标冲突 → 交人工，不自扩权 |

双层架构：**Loop 层**（`AgentLoop` 单目标收敛）→ **Graph 层**（`run_graph` 调度与 handoff）。
实现见同目录 `agent_loop.py`。

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../../.."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("REPO_ROOT =", REPO_ROOT)

In [ ]:
from utils.std_model import base_model
from agent_loop import run_graph, python_code_verifier

llm = base_model()

## 示例：带客观验证器的 Loop（写代码 + 真正执行测试）

行动 agent 写出 `fib(n)`，评测器拿到「代码执行结果」做客观判定。

In [ ]:
aim = (
    "写一个 Python 函数 fib(n)，返回斐波那契数列第 n 项（n 从 0 开始，"
    fib(0)=0, fib(1)=1）。要求函数名为 fib，并能正确计算 fib(10)。"
)

result = run_graph(aim, llm, verifier=python_code_verifier, budget=3)

print("--- Loop 历史 ---")
for h in result["history"]:
    print(f"  第 {h['iter']} 轮: {h['verdict']} — {h['reason']}")
print("\n最终状态:", result["status"])
print("Graph 决策:", result["graph_decision"])
print("\n--- 最终产出 ---")
print(result["final_output"])

## 观察预算耗尽（故意设 budget=1 且目标难以一次达成）

把 budget 设为 1，并给一个容易出错的复杂任务，即可看到 `budget_exhausted` 的诚实停止信号。

In [ ]:
result2 = run_graph(
    "写一个处理大型 CSV 并做复杂分组聚合的 Python 脚本，要求含单元测试。",
    llm,
    budget=1,
)
print("状态:", result2["status"], "| Graph 决策:", result2["graph_decision"])